# Random Forest: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains a random forest to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, following the EDA's decisions
3. trains a first random forest and checks it on the calibration split
4. tunes the forest's settings with a randomized search over time-based folds
5. saves the best model

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`, `SimpleImputer`), the model (`RandomForestClassifier`), the tuning tool (`RandomizedSearchCV`) and the scoring metrics
- **time / joblib:** timing the search and saving the final model to a file

In [26]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [23]:
train = pd.read_parquet("modeling_train.parquet")
calib = pd.read_parquet("modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

display(X_train_raw.head())
display(X_calib_raw.head())

display(y_train.head())
display(y_calib.head())

Training shape: (169827, 93)
Calibration shape: (297009, 93)


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,1050A093F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,698.0,30
1,1050A0ANF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
2,1080A06JF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,6.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
3,1080A075F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,7.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,681.0,30
4,1080A09LF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,14A0A001F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.0,6
1,14B0A005F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,139.0,7
2,1PG6SGEV,22,100.0,100.0,0.0,0.0,100.0,100.0,8.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.0,5
3,1PG700ZV,22,100.0,100.0,0.0,0.0,100.0,100.0,10.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.0,5
4,1PG73SEV,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,171.0,8


0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

## 3. Building the Model Inputs
Defines how the raw table is turned into model inputs. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [ ]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")          
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `rf_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:** fills blanks with the **training** median and adds a "was missing" column for each column that had blanks. The blanks come from specific drive models, such as S197 on two Toshiba models, so the fact that a value is missing is itself useful.
   - **vendor:** one-hot encoded

Because this sits inside a pipeline, the medians are learned from training data only. They're then re-used on the calibration split, so no information leaks from calibration into training.

In [16]:
rf_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First Random Forest (Before Tuning)
Builds the full model, preprocessing followed by a random forest, with reasonable starting settings:

- **`n_estimators=300`:** 300 trees, whose votes are averaged
- **`max_features="sqrt"`:** each split looks at a random subset of columns, which makes the trees different from each other
- **`min_samples_leaf=2`:** each leaf must contain at least 2 windows, to limit memorizing
- **`class_weight="balanced_subsample"`:** gives failures more weight, recalculated for each tree's sample of the data
- **`random_state=42`:** makes results repeatable
- **`n_jobs=-1`:** uses all CPU threads

In [17]:
rf_model = Pipeline([
    ("preprocessing", rf_preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing learns its medians, then the forest grows its 300 trees.

In [18]:
rf_model.fit(X_train_raw, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](93,)","['serial_number','window','smart_10_normalized_first',..., 'smart_9_normalized_delta','smart_9_raw_delta','drive_days_to_date']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,93
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the tr

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**Result:**
- **Ranking:** PR-AUC is **0.0995**, against 0.0073 for the SMART 5 rule, 0.0011 for random and 0.082 for logistic regression. ROC-AUC is 0.884.
- **Thresholds:** at 0.9, precision is 17.6% and recall 20.1%, the best F1 (0.188). At the default 0.5, precision is only 5.0%. The probabilities are too high because training has 43× more failures than reality, so a threshold has to be chosen deliberately and the probabilities recalibrated later.

In [21]:
results=[]
y_prob = rf_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "Random Forest",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,Random Forest,0.0995,0.8841,0.0011,1.0000,0.0022,0.0119
1,0.1,Random Forest,0.0995,0.8841,0.0093,0.7744,0.0184,0.0119
2,0.2,Random Forest,0.0995,0.8841,0.0188,0.7195,0.0367,0.0119
3,0.3,Random Forest,0.0995,0.8841,0.0321,0.6860,0.0613,0.0119
4,0.4,Random Forest,0.0995,0.8841,0.0407,0.6524,0.0766,0.0119
5,0.5,Random Forest,0.0995,0.8841,0.0500,0.6159,0.0926,0.0119
6,0.6,Random Forest,0.0995,0.8841,0.0617,0.5732,0.1115,0.0119
7,0.7,Random Forest,0.0995,0.8841,0.0811,0.5183,0.1403,0.0119
8,0.8,Random Forest,0.0995,0.8841,0.1138,0.4085,0.1780,0.0119
9,0.9,Random Forest,0.0995,0.8841,0.1765,0.2012,0.1880,0.0119


## 6. Tuning the Random Forest
### 6.1 Search Setup and Time-Based Folds
Sets up three things for tuning:

- **`rf_pipeline`:** the same preprocessing plus a forest whose settings will be searched (`class_weight="balanced"`)
- **`param_distributions`:** the settings to try:
  - number of trees
  - maximum tree depth
  - minimum windows needed to split a node, and to form a leaf
  - share of columns tried at each split
  - maximum number of leaves

  Together that's 864 combinations.
- **`time_folds()`:** builds 4 cross-validation folds **by time**. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead. This mirrors real use: predicting the future from the past.

The printout shows the 4 folds growing: training uses more months each time, and each validation block covers the next few months.

In [24]:
rf_pipeline = Pipeline([
    ("preprocess", rf_preprocess),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",   # failures are 1 in 21 in train: give them more weight
        n_jobs=-1,                 # use all 8 threads for each forest
        random_state=42,
    )),
])

param_distributions = {
    "classifier__n_estimators": [300, 500, 800],
    "classifier__max_depth": [10, 20, 30, None],
    "classifier__min_samples_split": [2, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__max_features": ["sqrt", 0.5],
    "classifier__max_leaf_nodes": [None, 100, 300]
}

## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")


fold 0: train 28,508 rows -> validate 37,027 rows
fold 1: train 64,596 rows -> validate 31,106 rows
fold 2: train 95,671 rows -> validate 32,768 rows
fold 3: train 128,057 rows -> validate 33,501 rows


### 6.2 Randomized Search, Results and Saving the Best Model
- **Randomized search:** tries **20 random combinations** from the 864. Each is trained once per fold, giving **80 fits**, and scored by PR-AUC (`average_precision`). `n_jobs=1` trains one forest at a time, because each forest already uses every CPU thread and several large forests at once could run out of the 12 GB of memory.
- **Refit:** the best combination is retrained on all of the training data (`refit=True`).
- **Results table:**
  - `mean_test_score`: the average PR-AUC over the 4 folds
  - `std_test_score`: how much the 4 fold scores vary
  - `rank_test_score`: 1 = best

  The table is saved to `rf_search_results.csv`.
- **Save:** the best model is saved to `rf_best.joblib`, so it can be reloaded without retraining.

**Result:**
- **Time:** the search took **36 minutes**.
- **Best settings:** **500 trees, no depth limit, `min_samples_leaf=10`, `min_samples_split=20`, `max_features="sqrt"`**, with a mean time-CV PR-AUC of **0.573** (std 0.032).
- **Close calls:** the top 6 settings are within 0.004 of each other, less than one std apart, so they perform about the same. The 6th-ranked setting (300 trees, depth 10) trains in about 7 seconds per fold against 17, which would be a faster alternative.

These scores come from training folds, where failures are 1 in 21, so they're much higher than PR-AUC on the calibration split. Use them only to compare settings with each other.

In [27]:
search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=20,                     
    scoring="average_precision",   
    cv=cv_folds,
    n_jobs=1,                                    
    refit=True,                    
    random_state=42,
    verbose=2,
    return_train_score=False,
)

start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

results = (pd.DataFrame(search.cv_results_)
           .sort_values("rank_test_score")
           [["rank_test_score", "mean_test_score", "std_test_score", "mean_fit_time"]
            + [c for c in search.cv_results_ if c.startswith("param_")]])
display(results.head(10))
results.to_csv("rf_search_results.csv", index=False)

#Save the best model (so a restart doesn't cost a retrain)
best_rf = search.best_estimator_
joblib.dump(best_rf, "rf_best.joblib")


Fitting 4 folds for each of 20 candidates, totalling 80 fits
[CV] END classifier__max_depth=10, classifier__max_features=0.5, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=5, classifier__min_samples_split=2, classifier__n_estimators=300; total time=   4.8s
[CV] END classifier__max_depth=10, classifier__max_features=0.5, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=5, classifier__min_samples_split=2, classifier__n_estimators=300; total time=  11.9s
[CV] END classifier__max_depth=10, classifier__max_features=0.5, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=5, classifier__min_samples_split=2, classifier__n_estimators=300; total time=  19.3s
[CV] END classifier__max_depth=10, classifier__max_features=0.5, classifier__max_leaf_nodes=300, classifier__min_samples_leaf=5, classifier__min_samples_split=2, classifier__n_estimators=300; total time=  29.4s
[CV] END classifier__max_depth=None, classifier__max_features=sqrt, classifier__max_leaf_nodes=

,rank_test_score,mean_test_score,std_test_score,mean_fit_time,param_classifier__n_estimators,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_leaf_nodes,param_classifier__max_features,param_classifier__max_depth
14,1,0.572895,0.031685,17.201931,500,20,10,None,sqrt,None
13,2,0.572582,0.031396,26.832961,800,10,10,None,sqrt,30
11,3,0.571806,0.032118,52.279824,500,10,5,None,0.5,None
0,4,0.571340,0.031077,16.256150,300,2,5,300,0.5,10
4,5,0.569712,0.027836,16.586034,800,10,5,None,sqrt,10
19,6,0.569588,0.028198,6.837530,300,20,5,300,sqrt,10
12,7,0.561873,0.033216,45.485468,500,2,5,300,0.5,30
2,8,0.561724,0.032831,68.042255,800,10,5,300,0.5,30
9,9,0.560896,0.034847,44.853674,500,10,2,300,0.5,30
16,10,0.560541,0.025002,10.971341,500,10,10,100,sqrt,10


['rf_best.joblib']